# Event Loop
* The event loop is the core programming mechanism that manages and schedules asynchronous tasks within a single thread.

* It continuously runs (conceptually like a `while True` loop), monitoring tasks, I/O operations, timers, and other events.

* Whenever a task is ready to execute, the event loop schedules it to run. If a task needs to wait for an external operation (such as a network request or file read), the event loop temporarily switches to another ready task instead of blocking the thread.

* When the waiting operation completes, the event loop schedules that task to continue its execution.

* This enables a single thread to efficiently handle many I/O-bound tasks concurrently without creating multiple threads.

# Routine (Synchronous Function)
A routine is simply a normal function that executes from start to finish without pausing in the middle.
* A routine is a normal function that executes sequentially from the first statement to the last.

* Once it starts executing, it continues until it finishes or returns a value.

* If the routine performs a slow operation (such as reading a file or making an API call), the entire program waits until that operation completes.

* In Python, routines are defined using the `def` keyword.



```
def greet():
   print("Hello")
   a = 5+3
    print("Welcome")

`greet()
```
````
Start
  ↓
print("Hello")
  ↓
print("Welcome")
  ↓
End
```

# Coroutine
A coroutine is a special kind of function designed for asynchronous programming.
* It is a special function that can temporarily pause its execution and resume later from the same point.

* While a coroutine is paused, the event loop can schedule and execute other ready tasks.

* This allows multiple I/O-bound operations to make progress efficiently without blocking the thread.

* In Python, coroutines are defined using the `async def` keyword.

* Example
 ```
async def fetch_data():
    print("Fetching data...")
```

| Routine                                | Coroutine                                       |
| -------------------------------------- | ----------------------------------------------- |
| Defined using `def`                    | Defined using `async def`                       |
| Executes continuously until completion | Can pause and resume execution                  |
| Used for synchronous programming       | Used for asynchronous programming               |
| Blocks during slow I/O operations      | Allows other tasks to run while waiting for I/O |
| Runs directly when called              | Must be scheduled/run by the event loop         |


# await

* The `await` keyword is used inside an asynchronous function (coroutine) to pause its execution until an awaitable operation completes.

* While the coroutine is paused, it yields control back to the event loop, allowing other ready tasks to execute instead of blocking the thread.

* Once the awaited operation completes, the event loop resumes the coroutine from where it paused.

* `await` can only be used inside an `async def` function.
 ```
import asyncio

async def fetch_data():
    print("Fetching data...")
    await asyncio.sleep(2)
    print("Data received")

async def main():
    await fetch_data()

asyncio.run(main())
  ```

# Task

* A Task is an object that represents the execution of a coroutine in the event loop.

* When a coroutine is converted into a Task, the event loop can schedule and execute it independently.

* Tasks allow multiple asynchronous operations to run concurrently within the same thread.

* Tasks are created using asyncio.create_task().

```
import asyncio


async def prepare_pizza():

    print("🍕 Preparing pizza")

    await asyncio.sleep(3)

    print("🍕 Pizza ready")


async def main():

    pizza_task = asyncio.create_task(
        prepare_pizza()
    )

    print("Customer can do other things")

    await pizza_task


asyncio.run(main())
```

# Task Example for GenAI

 ## Imagine a chatbot needs:
  * Retrieve user history
 * Search vector database
 * Get user preferences

  Without tasks:

  * History API - 2 seconds
  * Vector Search - 3 sec
  * Preferences - 1 sec
   
    Total = 6 sec
    
    It executes and completes in (2+3+1) 6 seconds to get required data
With tasks
* History API - 2 seconds
  * Vector Search - 3 sec
  * Preferences - 1 sec

   
   Total = 3 it executes and completes with the maximun time taken by the slowest task in the group of tasks


  To wait for all these tasks and get results we use asyncio.gather()
  
```
history_task = asyncio.create_task(
    get_history()
)

search_task = asyncio.create_task(
    vector_search()
)

prefs_task = asyncio.create_task(
    get_preferences()
)
await pizza_task
await burger_task
```

* Note: the above way of creating tasks  and  the below
the max time to gets results will be equal to slowest task
below one is the more optimised way of writing
```
results = await asyncio.gather(
    get_history(),
   vector_search(),
    get_preferences()
)``

# How the Event Loop + Coroutines + await Work Together

Imagine you're at a busy restaurant.
### Asyncio Analogy: The Restaurant

| Characters | Python Concept |
| :--- | :--- |
| **Restaurant Analogy** | Event Loop |
| **Waiter** | Coroutine |
| **Customer's order** | `await` |
| **Customer waiting for food** | I/O Operation |
| **Chef preparing the food** | Ready Task |
| **Customer whose food is ready** | *Result / Callback* |

Step 1: Customers Place Their Orders (Coroutines)

  Three customers enter the restaurant. and place orders


  Customer 1 → Pizza
  Customer 2 → Burger
  Customer 3 → Pasta

* Step 2: Waiter Takes Orders (Event Loop)

* Step 3: Customer Starts Waiting (await)
  The waiter gives Customer A's order to the chef.

  Cooking takes 10 minutes.

Instead of standing beside the kitchen doing nothing, the waiter serves other customers.

* Step 4: Waiter Serves Other Customers

  While Pizza is cooking...

  the waiter

  takes sandwich order
  serves Burger
  brings drinks
  collects bills

In [ ]:
import asyncio
import time

# Customer 1 places an order
async def order_pizza():
    print("🍕 Customer 1: Pizza order placed.")

    # Chef starts cooking.
    # This coroutine pauses and returns control to the Event Loop.
    await asyncio.sleep(3)

    print("🍕 Customer 1: Pizza served!")


# Customer 2 places an order
async def order_burger():
    print("🍔 Customer 2: Burger order placed.")

    # Chef starts cooking.
    # This coroutine pauses and returns control to the Event Loop.
    await asyncio.sleep(2)

    print("🍔 Customer 2: Burger served!")
  # Customer 3 places an order
async def order_sandwich():
    print("🍔 Customer 3: sandiwch order placed.")

    # Chef starts cooking.
    # This coroutine pauses and returns control to the Event Loop.
    await asyncio.sleep(2)

    print("🍔 Customer 3: sandiwch served!")


async def main():
    start = time.time()

    print("👨‍🍳 Restaurant opens...\n")

    # Register both coroutines as Tasks in the Event Loop
    pizza_task = asyncio.create_task(order_pizza())
    burger_task = asyncio.create_task(order_burger())
    sandwich_task = asyncio.create_task(order_sandwich())

    print("👨‍🍳 Waiter has registered both orders with the Event Loop.\n")

    # Wait for both tasks to complete
    await pizza_task
    await burger_task
    await sandwich_task
    print(f"\n⏱️ Total Time: {time.time() - start:.2f} seconds")


# Creates and starts the Event Loop
await main()

In [ ]:
import asyncio
import time

async def wash_dishes():
    print("1. Started washing dishes...")
    # The coroutine yields control back to the event loop here
    await asyncio.sleep(2)
    print("1. Finished washing dishes!")

async def boil_water():
    print("2. Started boiling water...")
    # The coroutine yields control back to the event loop here
    await asyncio.sleep(1)
    print("2. Water is boiling!")

async def main():
    # asyncio.gather registers both tasks into the event loop
    start_time = time.time()
    await asyncio.gather(wash_dishes(), boil_water())
    print(f"Total time taken: {time.time() - start_time:.2f} seconds")

# This call initializes, runs, and closes the event loop automatically
await main()

# Semaphore

* A Semaphore is a synchronization mechanism that controls the number of tasks that can access a resource at the same time.

* It maintains a counter called permits. Each task must acquire a permit before entering a restricted section of code.

* When all permits are used, new tasks wait until an existing task releases a permit.

* In asyncio, Semaphore is used to limit concurrency and prevent overwhelming external resources such as APIs, databases, or services.

Restaurant Analogy

Imagine a Restaurant

Restaurant has 5 tables
20  customers group are waiting outside
Customers  group can enter whenever a seat becomes available

"Only 5 customers can enter the restaurant. Others must wait." or one table frees next available customer can go inside

```
Restaurant capacity = 5

Inside:

Customer 1
Customer 2
Customer 3
Customer 4
Customer 5


Waiting:

Customer 6
Customer 7
Customer 8


Customer 2 finishes quickly.

Inside:

Customer 1
Customer 3
Customer 4
Customer 5


Empty seat:

1
Customer 6 enters

Inside:

Customer 1
Customer 3
Customer 4
Customer 5
Customer 6

The restaurant is back to capacity 5.
```

In [ ]:
import asyncio
import random

# Restaurant capacity = 5 customers
embedding_limit = asyncio.Semaphore(5)


async def create_embedding(chunk_id):

    async with embedding_limit:

        print(
            f"Embedding chunk {chunk_id}"
        )

        # Simulating embedding API call
        number = random.uniform(1, 4)
        await asyncio.sleep(number)

        print(
            f"Completed chunk {chunk_id}"
        )

        return f"embedding_{chunk_id}"


async def main():

    chunks = range(20)


    tasks = []

    for chunk in chunks:

        task = asyncio.create_task(
            create_embedding(chunk)
        )

        tasks.append(task)


    embeddings = await asyncio.gather(
        *tasks
    )


    print(
        f"Total embeddings created: {len(embeddings)}"
    )


await main()